# Intro - Modul Pendul

Se hjemmeside samt vejledningen for forklaringer, opgaver, eksempler. Denne fil er tiltænkt dem der gerne vil se hvad der foregår inde bag hjemmesiden, samt have mere frihed til at ændre (hyper)parametre. Den kræver at du er fortrolig med python.

Før vi går i gang, skal vi have hentet nogle "pakker". Tænk på det som nogen der har skrevet noget kode vi kan importere, så vi ikke skal gøre alting fra bunden. Når du kører cellen herunder, importerer og installerer den automatisk det vi skal bruge.

In [1]:
# Importer pakker der skal bruges
#%pip install -q numpy matplotlib scikit-learn lightgbm graphviz ipython scipy openpyxl pandas

# Data
import numpy as np
import scipy as scipy

# Plotting
import matplotlib.pyplot as plt

# Sklearn: et librabry med en masse funtioner vi bruger i Machine Learning
import sklearn as sklearn

# LightGBM - pakke til at køre decision tree
import lightgbm as lgb
from lightgbm import early_stopping

#Kompliceret ligning
from scipy.special import ellipk

#Vis NN struktur
from graphviz import Digraph
from IPython.display import display

#PDP
from sklearn.inspection import PartialDependenceDisplay
from sklearn.pipeline import Pipeline
import matplotlib.lines as mlines


## Data

Først vil vi gerne undersøge hvilken data vi har med at gøre. 

In [2]:
data = np.genfromtxt(r"C:\Users\beego\Desktop\Deployment website\data\pendulum_data_long.csv",
                     delimiter=',', names=True, dtype=None, encoding=None)

variabler = data.dtype.names
input_variabler = [v for v in variabler if v != 'Period']
input_data = np.column_stack([data[v] for v in input_variabler])
truth_data = data['Period']

# Print first 5 rows with names
print(" | ".join(variabler))
for row in data[:5]:
    print(" | ".join(str(row[v]) for v in variabler))

L_measured | alpha | theta0 | crosssection | m_total | Period
0.8747302965096528 | 0.9507143064099162 | 1.1247915185359671 | 0.09381218778758511 | 0.3248149123539492 | 1.717383880819979
1.473603804647257 | 0.020584494295802447 | 1.457873793026792 | 0.12654196971205903 | 0.36987128854262097 | 2.8178697965358013
0.7467799451988155 | 0.5247564316322378 | 0.704723026098962 | 0.05077207962772587 | 0.6894823157779035 | 1.669463972884501
0.8583625524795948 | 0.45606998421703593 | 1.199246345950219 | 0.03795432950217036 | 0.6113875507308892 | 1.9407698682697936
1.2923802257223773 | 0.17052412368729153 | 0.19107223017939132 | 0.14284397521546666 | 0.9725056264596474 | 2.2675931389942074


I cellen nedenfor kan du vælge hvor stor en andel af data du vil bruge til at træne modellen.

In [3]:
# Vælg hvor stor andel. Tallet skal være mellem 0 og 1.0.
andel_af_data = 1.0

Vi resampler vores data til kun at indeholde den ovenbestemte andel og klargør denne til brug i modellen.

In [4]:
#Brug valgt andel
input_data_justeret, truth_data_justeret = sklearn.utils.resample(
    input_data, truth_data, 
    n_samples=int(andel_af_data * len(input_data)), 
    random_state=42, 
    replace=False
    )
# Tilrettelæg til brug i model
data_træning, data_test, sand_periode_træning, sand_periode_test = \
    sklearn.model_selection.train_test_split(input_data_justeret, truth_data_justeret, test_size=0.25, random_state=42)

## Det neurale netværk

For at data kan bruges i det neurale netværk er vi nødt til at skalere det således at det ligger fordelt omkring 0 (der er ~lige mange værdier over og under 0) og dets spredning er 1 (værdierne ligger tæt på 0).

In [5]:
scaler = sklearn.preprocessing.StandardScaler()

data_træning_scaled = scaler.fit_transform(data_træning)
data_test_scaled = scaler.transform(data_test)

I et neuralt netværk kan vi justere på hvor mange lag og hvor mange noder hvert lag skal have:

In [6]:
layer_one   = 4            # Her definerer vi hvor mange lag der skal være, og hvor mange 'neuroner' eller noder 
layer_two   = 2            # hvert lag skal have. Prøv at ændre på antal noder 
layer_three = 2             # for at forbedre din model
layer_four  = 2
layer_five  = 2
layer_six   = 2

Nedenfor træner vi modellen. Vi kan også regne ud hvor mange parametre modellen bruger.
Herefter plotter vi for at se hvor godt modellen klarer sig.

In [ ]:
# Her definerer og træner vi modellen
mlp = sklearn.neural_network.MLPRegressor(hidden_layer_sizes=(layer_one, layer_two, layer_three, layer_four, layer_five, layer_six), 
max_iter=500, early_stopping=True, random_state=42)
mlp.fit(data_træning_scaled, sand_periode_træning) 

# Her giver vi den trænede model test data som den ikke har set før, og beder om at forudsige dybden
forudsagt_periode = mlp.predict(data_test_scaled)  

# Beregn antal parametre i modellen
# Coef er vægtene er intercept er bias. Den henter antallet direkte fra modellen.
n_params = sum(coef.size + intercept.size for coef, intercept in zip(mlp.coefs_, mlp.intercepts_))
print(f"Antal parametre i NN: {n_params}")

#Print info omkring træningens forløb
print("n_iter:", mlp.n_iter_)          # actual iterations performed
print("max_iter:", mlp.max_iter)
if mlp.n_iter_ < mlp.max_iter:
    print("MLPRegressor stopped early")
print("loss_curve (last values):", mlp.loss_curve_[-5:])
if hasattr(mlp, "validation_scores_"):
    print("validation_scores (last):", mlp.validation_scores_[-5:])

Kør nedenstående celle for at visualisere netværket i modellen.

In [ ]:
# -----------------------------
# Funktion til at hente lagstørrelser
# -----------------------------
def get_layer_sizes(mlp_model, input_dim, output_dim):
    """
    Returnerer en liste med antal noder per lag:
    [input, hidden1, hidden2, ..., output]
    """
    hidden = list(mlp_model.hidden_layer_sizes)
    return [input_dim] + hidden + [output_dim]

# Hent lagstørrelser fra din trænede model
layer_sizes = get_layer_sizes(
    mlp_model=mlp,
    input_dim=len(input_variabler),
    output_dim=1 # 1 fordi vi forudsiger en enkelt kontinuerlig værdi (Periode)
)

print("Layer sizes:", layer_sizes)

# -----------------------------
# Funktion til at tegne netværket
# -----------------------------
def draw_sklearn_mlp(
    layer_sizes,
    feature_names=None,
    class_names=None,
    max_visible_nodes_per_layer=32,
    filename="sklearn_mlp_architecture",
    format="png"
):
    dot = Digraph("NeuralNetwork", format=format)
    dot.attr(rankdir="LR", splines="line", nodesep="0.35", ranksep="1.0")
    dot.attr("node", shape="circle", fixedsize="true", width="0.45", fontsize="10")

    layer_node_ids = []

    for layer_idx, size in enumerate(layer_sizes):
        current_layer_ids = []

        # Bestem hvor mange noder der skal vises for at undgå at plottet bliver for stort
        visible = min(size, max_visible_nodes_per_layer)

        if size > max_visible_nodes_per_layer:
            top_n = max_visible_nodes_per_layer // 2
            bottom_n = max_visible_nodes_per_layer - top_n
            visible_indices = list(range(top_n)) + list(range(size - bottom_n, size))
            use_ellipsis = True
        else:
            visible_indices = list(range(size))
            use_ellipsis = False

        with dot.subgraph() as s:
            s.attr(rank="same")

            # Titel til laget
            if layer_idx == 0:
                layer_label = f"Input\n({size})"
            elif layer_idx == len(layer_sizes) - 1:
                layer_label = f"Output\n({size})"
            else:
                layer_label = f"Hidden {layer_idx}\n({size})"

            title_id = f"layer_title_{layer_idx}"
            s.node(title_id, label=layer_label, shape="plaintext", fontsize="12")

            # Synlige neuroner
            for neuron_idx in visible_indices:
                node_id = f"L{layer_idx}_N{neuron_idx}"

                if layer_idx == 0 and feature_names is not None and neuron_idx < len(feature_names):
                    label = feature_names[neuron_idx]
                    shape = "box"
                elif layer_idx == len(layer_sizes) - 1 and class_names is not None and neuron_idx < len(class_names):
                    label = class_names[neuron_idx]
                    shape = "box"
                else:
                    label = ""
                    shape = "circle"

                s.node(node_id, label=label, shape=shape)
                current_layer_ids.append(node_id)

            # Tilføj punktummer for de skjulte noder
            if use_ellipsis:
                ellipsis_id = f"L{layer_idx}_ellipsis"
                s.node(ellipsis_id, label="...", shape="plaintext", fontsize="18")
                insert_pos = len(current_layer_ids) // 2
                current_layer_ids.insert(insert_pos, ellipsis_id)

        layer_node_ids.append(current_layer_ids)

    # Forbind lagene med edges
    for i in range(len(layer_node_ids) - 1):
        src_nodes = layer_node_ids[i]
        dst_nodes = layer_node_ids[i + 1]

        src_real = [n for n in src_nodes if "ellipsis" not in n]
        dst_real = [n for n in dst_nodes if "ellipsis" not in n]

        for src in src_real:
            for dst in dst_real:
                dot.edge(src, dst)

    #output_path = dot.render(filename=filename, cleanup=True)
    return dot

# -----------------------------
# Generer diagram
# -----------------------------
graph = draw_sklearn_mlp(
    layer_sizes=layer_sizes,
    feature_names=input_variabler,
    class_names=["Periode"],
    max_visible_nodes_per_layer=6,
    filename="pendul_mlp_arkitektur",
    format="png"
)

display(graph)
#print(f"Diagram gemt til: {output_file}")

Evaluer modellens performance ved at plotte modellens forudsagte perioder mod de sande perioder.

In [ ]:
#Make a simple true vs predicted plot
def plotting(sand, forudsagt):
    plt.figure(figsize=(8, 8))
    plt.scatter(sand, forudsagt, alpha=0.5)
    plt.plot([min(sand), max(sand)], [min(sand), max(sand)], color='red', linestyle='--')
    plt.xlabel('Sand værdi')
    plt.ylabel('Forudsagt værdi')
    plt.title('Sand vs Forudsagt')
    plt.grid()
    plt.show()
plotting(sand_periode_test, forudsagt_periode)

Residualplottet visualiserer hvor mange punkter ligger så og så langt væk fra den korrekte forudsigelse.

In [ ]:
residuals = sand_periode_test-forudsagt_periode
fig3, ax3 = plt.subplots(figsize=(8, 6))
ax3.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
ax3.axvline(x=0, color='red', linestyle='--', label='Perfekt forudsigelse')
ax3.set_xlabel("Sand periode − Forudsagt periode")
ax3.set_ylabel("Antal")
ax3.set_title("Histogram over residualer")
ax3.legend()
ax3.grid(True)
fig3.tight_layout()
plt.show()

Vi undersøger hvilke parametre der har størst betydning for modellens forudsigelse ved at udføre permutation importance.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names, but .* was fitted with feature names")

res = sklearn.inspection.permutation_importance(mlp, data_test_scaled, sand_periode_test, scoring="neg_mean_squared_error")

imp_mse = res.importances_mean                
order = np.argsort(imp_mse)[::-1]
labels = np.asarray(variabler[:-1])[order]
vals = imp_mse[order]

plt.figure(figsize=(8, 6))
y = np.arange(len(vals))
plt.barh(y, vals)
plt.yticks(y, labels)
plt.xlabel("Increase in MSE (permutation)")
plt.ylabel("Feature")
plt.title("Permutation Importance")
plt.gca().invert_yaxis()  
plt.tight_layout()
plt.show()

Visualiser modellens loss vs iterationer. Dette visualiserer hvordan modellen optimerer vægtene for hver iteration der går.

In [ ]:
fig_loss, ax_loss = plt.subplots(figsize=(5, 5))
ax_loss.plot(mlp.loss_curve_, color='tab:blue', linewidth=2)
ax_loss.set_xlabel("Iterations")
ax_loss.set_ylabel("MSE Loss")
ax_loss.set_title("MSE Loss vs Iterations")
ax_loss.grid(True)
plt.show()


Lad modellen forudsige perioder ud fra dine egne værdier. 
Cellen nedenfor laver en forudsigelse af perioden på dine data og plotter dine målinger mod dens forudsigelser (også på dine data). Linjen viser hvor data burde ligger hvis de stemte overens (den er ikke lavet ud fra data men bare tegnet oveni). 

Vælg om du vil tegne perioden udregnet fra dine data med den simple ligning og/eller om du vil medtage den komplicerede ligning.

In [ ]:
# For at sammenligne med beregning med small angle approximation 
# Sæt nedenstående til 1
Vis_Beregning_med_simpel_ligning = 1
Vis_Beregning_med_kompliceret_ligning = 1

In [14]:
#Cellen tager excel-skabelonen som input.
import pandas as pd
import numpy as np

#Indsæt din egen fil nedenfor
#Læg filen i samme mappe som denne notebook, skriv filnavnet ud fra dataset_path nedenfor
dataset_path = r"VORES_datasæt.xlsx"

df = pd.read_excel(dataset_path, skiprows=2, engine='openpyxl') #Understøtter nyere excelformater

# Number of features the model expects
n_features = getattr(mlp, "n_features_in_", None)

#Del data op i input og periode
X_new = df.iloc[:, :n_features].apply(pd.to_numeric, errors="coerce").to_numpy()
målt_periode = pd.to_numeric(df.iloc[:, n_features], errors="coerce").to_numpy()


In [15]:
#Bergen periode med kompliceret ligning
rho_air = 1.225        # kg/m^3
C_d = 1.1              # cylinder, transverse flow
g = 9.81

L, alpha, theta0, crosssection, m_total = X_new[:, 0], X_new[:, 1], X_new[:, 2], X_new[:, 3], X_new[:, 4]
shape = ((1/3)*alpha + (1 - alpha)) / ((1/2)*alpha + (1 - alpha))
T0 = 2 * np.pi * np.sqrt(L / g * shape)
k2 = np.sin(theta0 / 2)**2
angle_factor = (2 / np.pi) * ellipk(k2)
A = crosssection
omega0 = 2 * np.pi / T0

gamma = (rho_air * C_d * A * L) / m_total
drag_factor = 1 + (gamma / omega0)**2 / 8

T_komp =  T0 * angle_factor * drag_factor

In [ ]:
#Beregn periode med small angle approximation for målt længde.
g = 9.81  # m/s^2
L = X_new[:, 0]
T_simple = 2 * np.pi * np.sqrt(L / g)

#Beregn periode med kompliceret ligning


#Skaler input så vi kan bruge det i vores NN model
X_new_scaled = scaler.transform(X_new)
#Forudsig med modellen på vores egen input data 
forudsagt_periode_egen = mlp.predict(X_new_scaled)

#Regn usikkerheder
mae_model = np.mean(np.abs(forudsagt_periode_egen - målt_periode))
mae_simple = np.mean(np.abs(T_simple - målt_periode))
mae_komp = np.mean(np.abs(T_komp - målt_periode))
            
#Plot
plt.figure(figsize=(6, 6))
plt.scatter(målt_periode, forudsagt_periode_egen, alpha=0.5, label=f"Model (MAE={mae_model:.3f})", color='blue')

mn = float(np.min([målt_periode.min(), forudsagt_periode_egen.min(), T_simple.min(), T_komp.min()]))
mx = float(np.max([målt_periode.max(), forudsagt_periode_egen.max(), T_simple.max(), T_komp.max()]))
plt.ylabel("Forudsagt Periode")

if Vis_Beregning_med_simpel_ligning == 1:
    plt.scatter(målt_periode, T_simple, alpha=0.5, label=f"T=2π√(L/g) (MAE={mae_simple:.3f})", color='orange')
    plt.ylabel("Forudsagt / Beregnet Periode")

if Vis_Beregning_med_kompliceret_ligning == 1:
    plt.scatter(målt_periode, T_komp, alpha=0.5, label=f"Kompliceret ligning (MAE={mae_komp:.3f})", color='green')
    plt.ylabel("Forudsagt / Beregnet Periode")

plt.plot([mn, mx], [mn, mx], "r--", label="Perfekt overensstemmelse")
plt.xlabel("Målt Periode")
plt.title("Målt vs Forudsagt Periode")
plt.grid(True)
plt.legend()
plt.show()
  


##### Partial Dependence Display
Forklaring og opgaver til dette findes i vejledningen under afsnit 3.3. 

In [ ]:
# 1. Saml den allerede trænede scaler og model i en pipeline
pipeline = Pipeline([
    ('scaler', scaler),
    ('mlp', mlp)
])

features_to_plot = list(range(len(input_variabler)))

# 2. Generer Partial Dependence Plots
fig, ax = plt.subplots(figsize=(12, 8))
display = PartialDependenceDisplay.from_estimator(
    estimator=pipeline,       
    X=data_test,          
    features=features_to_plot,
    feature_names=input_variabler,
    kind="both",
    grid_resolution=10,
    ax=ax,
    subsample=1000,
    pd_line_kw={"color":"tab:blue","linestyle":"-","linewidth":2,"label":"."},
    ice_lines_kw = {"color": "lightblue", "alpha": 0.12, "linewidth": 0.5,"label":"."}
)
for ax in display.axes_.flat:
    # ax kan være None hvis grid'et ikke er fuldt udfyldt (f.eks. 5 plots i et 2x3 grid)
    if ax is not None:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

# Create custom legend items
ice_line = mlines.Line2D([], [], color='tab:blue', alpha=1,linewidth=2, label='Gennemsnitlig Partial Dependence')
pdp_line = mlines.Line2D([], [], color='lightblue',alpha=1, linewidth=1, label='Individuelle Partial Dependence')

# Place a single legend on the figure itself
fig.legend(handles=[ice_line, pdp_line], loc='lower right', bbox_to_anchor=(0.98, 0.1), ncol=1, fontsize=12)

# Adjust layout so subplots don't overlap with super title and bottom legend isn't cut off
fig.subplots_adjust(top=0.92, bottom=0.12, wspace=0.3, hspace=0.4)
plt.tight_layout()
plt.show()